# Hate-speech classifier: training

Trains the TF-IDF + linear SVM (`SGDClassifier`, hinge loss) used by the API server.

**What this model is.** The training data (`output_filtered.csv`, columns `tweet` and `hate_speech_count`) is a *general* hate-speech / offensive-language tweet corpus. It contains no antisemitism-specific labels, so the model detects hateful or offensive language in general, not antisemitism. See "Limitations" in the README.

**Methodology note.** The data is split into train and test sets *before* the minority class is upsampled, and only the training split is resampled. Upsampling before splitting puts copies of the same tweet in both splits and inflates the test score. An earlier version of this notebook did that and reported F1 = 81.9, which was optimistic.

In [ ]:
import re

import joblib
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.utils import resample

RANDOM_STATE = 42

In [ ]:
data = pd.read_csv("data/output_filtered.csv")
print("Data set", data.columns.tolist(), data.shape)
print(data["hate_speech_count"].value_counts())

Preprocessing. This **must** match `preproc()` in `backend/api/nlp.py`.

In [ ]:
CLEAN_RE = re.compile(r"(@[\w]+)|([^0-9A-Za-z \t])|(\w+:\/\/\S+)|^rt|http.+?")
HASHTAG_RE = re.compile(r"#\w+\s?")

def preproc(text):
    text = str(text).lower()
    text = HASHTAG_RE.sub("", text)
    return CLEAN_RE.sub("", text)

data["tweet"] = data["tweet"].apply(preproc)
data = data.drop_duplicates(subset="tweet")
data.head()

Split first (stratified), then balance **only the training set**.

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    data["tweet"], data["hate_speech_count"], test_size=0.2,
    stratify=data["hate_speech_count"], random_state=RANDOM_STATE)

train = pd.DataFrame({"tweet": x_train, "label": y_train})
train_maj = train[train.label == 0]
train_min = train[train.label == 1]
train_min_upsampled = resample(train_min, replace=True, n_samples=len(train_maj), random_state=RANDOM_STATE)
train_balanced = pd.concat([train_maj, train_min_upsampled]).sample(frac=1, random_state=RANDOM_STATE)
print(train_balanced["label"].value_counts())

In [ ]:
model = Pipeline([
    ("vect", CountVectorizer()),
    ("tfidf", TfidfTransformer()),
    ("nb", SGDClassifier(random_state=RANDOM_STATE)),
])
model.fit(train_balanced["tweet"], train_balanced["label"])

## Evaluation on the untouched (imbalanced) test set

In [ ]:
y_pred = model.predict(x_test)
print(classification_report(y_test, y_pred, target_names=["not hateful", "hateful"], digits=3))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred), display_labels=["not hateful", "hateful"]).plot();

## Known failure case

This post contains no hate speech or antisemitism, but the model flags it. It is kept here as an example of the false positives described in the README, not as a success.

In [ ]:
example = """Because sometimes, against all the odds, we keep the faith, and hope!
We will show solidarity because we know they need us. We will put pressure on the authorities
for ceasefire. We want genocide to stop. We will donate, we will learn the history.
Prayers and love for palestine and its people, always and forever!"""
print("Flagged:", bool(model.predict([preproc(example)])[0]))

In [ ]:
joblib.dump(model, "../backend/api/data/hate_speech_sgd.joblib")